# Time-to-event prediction with C-MAPSS

Time-to-event (TTE) estimates when a defined event will occur. Here the event
is engine failure and time is measured in operating cycles.
`pdmdata.tasks.tte.prepare_cmapss` builds a Polars-tabular bundle; RUL is the
same remaining-time quantity under an end-of-life event profile.

## Learning goals

- prepare row-level ``time_to_event`` with ``event_observed``;
- hold out engines for validation without leaking across units;
- contrast official remaining-time scoring with entity-level censoring.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import HistGradientBoostingRegressor
from pdmdata.tasks.tte import evaluate_test_predictions, prepare_cmapss
from pdmdata.tasks.metrics import regression_metrics

In [ ]:
pdmdata.download("cmapss")
bundle = prepare_cmapss(
    "FD001",
    time_cap=125,
    validation_fraction=0.2,
    random_state=0,
    drop_constant_features=True,
)
bundle.summary()

In [ ]:
bundle.train.frame.select(
    "unit_number", "cycle", "time_to_event", "event_observed"
).head()

In [ ]:
plt.hist(
    bundle.train.frame["time_to_event"].to_numpy(), bins=60
)
plt.xlabel("Operating cycles until failure")
plt.ylabel("Observations")
plt.show()

In [ ]:
model = HistGradientBoostingRegressor(
    loss="absolute_error", random_state=0
)
X_train, y_train, _ = bundle.train.to_numpy()
X_val, y_val, _ = bundle.validation.to_numpy()
model.fit(X_train, y_train)
print(regression_metrics(y_val, np.maximum(model.predict(X_val), 0)))

In [ ]:
full = prepare_cmapss(
    "FD001",
    time_cap=125,
    validation_fraction=0.0,
    drop_constant_features=True,
)
model.fit(*full.train.to_numpy()[:2])
prediction = np.maximum(
    model.predict(bundle.test_eval_split().X().to_numpy()), 0
)
print(evaluate_test_predictions(bundle, prediction))
entities = bundle.entity_table()
entities.group_by("cohort", "event_observed").len()

## TTE versus RUL and survival

Row-level ``time_to_event`` matches RUL when the event is end of useful life;
use `pdmdata.tasks.rul` when you want the ``RUL`` column name and NASA-oriented
helpers. ``entity_table()`` keeps test engines right-censored at the last
observed cycle (no official offset), which is the starting point for survival
analysis. See the survival-analysis notebook for Kaplan-Meier on that table.